In [1]:
from ingest import load_and_flatten_data
from rag import build_index, ask_assistant

documents = load_and_flatten_data()

print(f"Successfully processed {len(documents)} documents.")
print("\nSample document:")
documents[0]

Successfully processed 19 documents.

Sample document:


{'section': 'Origins',
 'question': "Who murdered Bruce Wayne's parents?",
 'text': 'Joe Chill is the mugger who murdered Thomas and Martha Wayne in Crime Alley, an event that drove Bruce to become Batman.',
 'category': 'Batman'}

In [2]:
minsearch_client = build_index(documents)

In [3]:
test_queries = [
    "Who is the street mugger responsible for the murder of Bruce's parents?",
    "What independent superhero identity did Dick Grayson take on when he grew up?",
    "What was the Joker's supposed profession according to theories about his past?",
    "What is the name of the massive secret base located beneath Wayne Manor?",
    "What is the name of the psychiatric hospital in Gotham where criminally insane villains are locked up?",
    "Who is Diana's mother and the queen of Themyscira?",
    "What is the magical weapon forged by Hephaestus that forces anyone bound by it to tell the truth?",
    "Who is the archaeologist villain cursed with a human-cheetah hybrid form?",
    "What was Superman's birth name on his home planet before it was destroyed?",
    "What radioactive fragment can drain Superman's powers and kill him?",
    # Hybrid / Multi-part questions
    "Who murdered Bruce Wayne's parents and who is the first Robin?",
    "Where is Wonder Woman from and what magical weapon does she use that forces truth?",
    "What is Superman's birth name and what radioactive fragment is his primary weakness?"
]

print(f"Running {len(test_queries)} test queries...\n")

for q in test_queries:
    answer = ask_assistant(q, minsearch_client)
    
    print(f"Question: {q}")
    print(f"Answer: {answer.strip()}")
    print("-" * 60)

Running 13 test queries...

Question: Who is the street mugger responsible for the murder of Bruce's parents?
Answer: Joe Chill.
------------------------------------------------------------
Question: What independent superhero identity did Dick Grayson take on when he grew up?
Answer: Nightwing.
------------------------------------------------------------
Question: What was the Joker's supposed profession according to theories about his past?
Answer: Failed stand-up comedian.
------------------------------------------------------------
Question: What is the name of the massive secret base located beneath Wayne Manor?
Answer: The Batcave.
------------------------------------------------------------
Question: What is the name of the psychiatric hospital in Gotham where criminally insane villains are locked up?
Answer: Arkham Asylum.
------------------------------------------------------------
Question: Who is Diana's mother and the queen of Themyscira?
Answer: Queen Hippolyta.
----------

In [4]:
advanced_queries = [
    # Typo test
    "Who is Joe Chil who murdered Thomas Wayne?",
    
    # Scattered keywords
    "parents of Bruce Wayne mugger shooter name",
    
    # Semantic / indirect search test
    "Which psychiatric facility holds Gotham's most dangerous inmates?",
    
    # Tricky questions containing distractors
    "Who trained or raised Diana and rules the island of Amazons?"
]

print(f"Running {len(advanced_queries)} test queries...\n")

for q in advanced_queries:
    answer = ask_assistant(q, minsearch_client)
    
    print(f"Question: {q}")
    print(f"Answer: {answer.strip()}")
    print("-" * 60)

Running 4 test queries...

Question: Who is Joe Chil who murdered Thomas Wayne?
Answer: Joe Chill.
------------------------------------------------------------
Question: parents of Bruce Wayne mugger shooter name
Answer: Joe Chill.
------------------------------------------------------------
Question: Which psychiatric facility holds Gotham's most dangerous inmates?
Answer: Arkham Asylum.
------------------------------------------------------------
Question: Who trained or raised Diana and rules the island of Amazons?
Answer: Queen Hippolyta.
------------------------------------------------------------


In [9]:
from agent import agent_loop

# The ULTIMATE instruction block for strict Agentic Behavior
instructions = """
You are an advanced investigative assistant specializing in DC Comics.
Your goal is to answer the user's question completely based on search results.

CRITICAL RULES:
1. USE EXACT KEYWORDS: When using the 'search' tool, construct your query using ONLY the concepts provided in the user's question. DO NOT invent or assume character names (e.g., do NOT add "Joker" if the user didn't say it).
2. PREVENT SCHEMA HALLUCINATION: When using the tool, ONLY output the search string. 
   -> Correct: {"query": "villain broke Batman back"}
   -> Incorrect: {"query": {"type": "string", "value": "villain broke Batman back"}}
3. MULTI-HOP SEARCHING: If the first search returns 'no information' or is irrelevant, you MUST call the search tool AGAIN with new keywords. 
4. DO NOT LEAK TOOL CALLS: NEVER write a JSON tool call as plain text in your final message. If you want to search, use the proper function calling mechanism.
5. If after multiple searches you still don't know, just say "I could not find the answer."
6. KEYWORD SEARCHING ONLY: Minsearch requires exact keyword matches. Do NOT send long sentences or multiple concepts to the search tool. Extract only 1 or 2 core entities (e.g., "Bane" or "Ace Chemicals" or "Red Hood") to maximize search success.
""".strip()

# Test 1: Typo Recovery (Location & Characters)
question = "What is the name of the aslyum where they keep the Jocker and Two-Fcae?"

print("Initializing Agent...")
final_answer = agent_loop(instructions, question, minsearch_client)

print("\n--- RESULT ---")
print(final_answer)

Initializing Agent...

[Iteration #1] Model is thinking...
 -> Executing Search: {"query":{"type":"string","value":"Joker asylum and Two-Face asylum"}}
 -> Sanitized query sent to database: 'Joker asylum and Two-Face asylum'

[Iteration #2] Model is thinking...
 -> Final Answer Generated.

--- RESULT ---
The asylum where they keep the Joker and Two-Face is called Arkham Asylum.


In [10]:
# Test 2: Typo Recovery (Event & Character)
question = "Who is the villian that brocke Batmen's bakc in the comics?"

print("Initializing Agent...")
final_answer = agent_loop(instructions, question, minsearch_client)

print("\n--- RESULT ---")
print(final_answer)

Initializing Agent...

[Iteration #1] Model is thinking...
 -> Executing Search: {"query":{"type":"string","value":"villain broke Batman back"}}
 -> Sanitized query sent to database: 'villain broke Batman back'

[Iteration #2] Model is thinking...
 -> Final Answer Generated.

--- RESULT ---
Based on the search results, I was unable to find information about a villain breaking Batman's back. However, I can tell you that Bane broke Batman's back in the comics. Bane is a supervillain in the DC Comics universe and is known for his physical strength and tactical genius. He broke Batman's back during their confrontation in "The Long Halloween" storyline.


In [11]:
# Test 3: Multi-Hop Reasoning
question = "What is the name of the mother of the superhero whose secret identity is Diana Prince?"

print("Initializing Agent...")
final_answer = agent_loop(instructions, question, minsearch_client)

print("\n--- RESULT ---")
print(final_answer)

Initializing Agent...

[Iteration #1] Model is thinking...
 -> Executing Search: {"query":{"type":"string"}}
 -> Sanitized query sent to database: 'string'

[Iteration #2] Model is thinking...
 -> Final Answer Generated.

--- RESULT ---
The search results did not provide any relevant information. I could not find the answer.


In [12]:
# Test 4: Specific Entity Extraction
question = "What is the exact name of the chemical plant where the Red Hood fell into a vat of acid to become the Joker?"

print("Initializing Agent...")
final_answer = agent_loop(instructions, question, minsearch_client)

print("\n--- RESULT ---")
print(final_answer)

Initializing Agent...

[Iteration #1] Model is thinking...
 -> Executing Search: {"query":{"type":"string","value":"Ace Chemicals"}}
 -> Sanitized query sent to database: 'Ace Chemicals'

[Iteration #2] Model is thinking...
 -> Final Answer Generated.

--- RESULT ---
The answer to the user's question is not in the search results. The chemical plant where the Red Hood fell into a vat of acid to become the Joker is actually Ace Chemicals Plant, but this event occurred with Jason Todd, not the original Red Hood (who was later resurrected as Red Hood).
